# Advanced Furnace ML：小数据预测与安全推荐

本实验使用前248炉开发、最后44炉锁定审计。开发区采用三个大时间折：**148/33、181/33、214/34**。模型与推荐规则冻结后才读取最后44炉。

推荐只有在保守预测低于相似历史实际气耗、至少2/3时间折同意、历史可行、不贴边且位于±10%信任区域时才通过。**离线预计节省不等于工厂实际节省，必须经过现场受控试验。**

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (cwd / 'advanced_furnace_ml' if (cwd / 'advanced_furnace_ml').exists() else cwd)
WORKSPACE_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from advanced_furnace_ml.artifacts import load_bundle, predict_bundle
from advanced_furnace_ml.data import FEATURE_COLS, load_batch_data
from advanced_furnace_ml.pipeline import run_advanced_pipeline

EXCEL_PATH = WORKSPACE_ROOT / '4_month_data_2026_02_01_2026_06_25.xlsx'
result = run_advanced_pipeline(
    EXCEL_PATH, PROJECT_ROOT, optimizer_budget=600, seeds=[0, 1, 2]
)
print('Project:', PROJECT_ROOT)
print('Selected before lock:', result['selected_before_lock'])
print('Locked batches:', len(result['locked_audit']))

## 1. 开发区模型矩阵与三时间折表现

In [ ]:
model_cv_summary = result['model_summary'].copy()
fold_metrics = pd.read_csv(PROJECT_ROOT / 'reports/chronological_fold_metrics.csv')
tree_tuning_summary = result['tree_tuning_summary'].copy()
random_cv_reference = result['random_cv_reference'].copy()
display(model_cv_summary.round(4))
print('LightGBM / CatBoost small-data parameter variants')
display(tree_tuning_summary.round(4))
print('Repeated random CV reference only (not used to select the chronological Champion)')
display(random_cv_reference.groupby('candidate', as_index=False).agg(random_rmse=('rmse', 'mean')).sort_values('random_rmse').round(4))
display(fold_metrics.sort_values(['fold', 'rmse']).round(4))
ax = model_cv_summary.head(10).sort_values('selection_score').plot.barh(
    x='candidate', y='selection_score', figsize=(9, 5), legend=False,
    title='Top development candidates: chronological selection score'
)
ax.set_xlabel('Mean RMSE + 0.25 × RMSE std')
plt.tight_layout()
plt.show()

## 2. 最后44炉锁定审计

下表只用于冻结后的最终审计，不反向修改开发区选择。

In [ ]:
locked_audit = result['locked_audit'].copy()
display(locked_audit.round(4))
selected_lock = locked_audit.loc[locked_audit['selected_before_lock']].iloc[0]
print(f"Frozen Champion locked RMSE: {selected_lock['rmse']:.2f}")
print(f"Frozen Champion 90% interval coverage: {selected_lock['interval_coverage_90']:.1%}")

## 3. 推荐模型、三个优化器与安全门

In [ ]:
recommendation_summary = result['recommendation_summary'].copy()
optimizer_runs = result['optimizer_runs'].copy()
optimizer_summary = optimizer_runs.groupby(['candidate', 'optimizer'], as_index=False).agg(
    median_best_objective=('best_objective', 'median'),
    mean_safe_candidate_rate=('safe_candidate_rate', 'mean'),
    evaluations=('evaluations', 'first'),
)
display(optimizer_summary.round(4))
display(recommendation_summary.round(4))
production = result['production_recommendation']
print('Production trial candidate / fallback:')
display(pd.DataFrame([production]))
print(result['bundle']['offline_savings_disclaimer'])

## 4. Joblib重载验证与最终结论

In [ ]:
bundle_path = PROJECT_ROOT / 'artifacts/advanced_furnace_bundle.joblib'
bundle = load_bundle(bundle_path)
df = load_batch_data(EXCEL_PATH)
smoke = predict_bundle(bundle, df[FEATURE_COLS].iloc[[0]])
display(smoke.round(4))
print('Prediction Champion:', bundle['selected_model_name'])
print('Recommendation model:', bundle['recommendation_model_name'])
print('Recommendation safety pass:', bundle['production_recommendation_86000']['safety_pass'])
print('offline_savings_disclaimer:', bundle['offline_savings_disclaimer'])
print('离线预计节省不等于工厂实际节省；实际效果必须通过现场受控试验。')